In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

from pathlib import Path
import re
import json
import shutil
import numpy as np
import pandas as pd
import xarray as xr

# ============================================================
# Auto time-fix + fill-to-365 (with NaN) + validation
#
# 自动识别 /mnt/soclim0/public_data/weiji 下的实验目录：
#   - 目录名不以 _timefixed 结尾
#   - 包含 U/V/T/OMEGA/PS/Z3/O3 这些子目录
#   - 子目录里存在 *.cam.h3.YYYY.VAR.nc 文件
#
# 对每个识别到的实验：
#   1) 复制到 *_timefixed
#   2) 对文件名年份 >= RUN2_START_YEAR 的文件修正内部 time/date/time_bnds
#   3) 对 ntime != 365 的年文件补齐到 365，新增数据全部 NaN
#   4) 做完整验证并写 marker
#
# 断点续跑 / 自动跳过：
#   - 若 OUT_ROOT/.TIMEFIX_AND_FILL_DONE 存在，则整个实验跳过
#   - 若已经 timefix 但未 fill，会只继续 fill+validate
#   - 若某个输出文件已存在且非空，会逐文件跳过
#
# 后续新增实验时：
#   - 只要目录结构一致，脚本会自动识别
#   - 若你想限制只处理某些实验，可设置 INCLUDE_PATTERNS
# ============================================================

BASE_ROOT = Path("/mnt/soclim0/public_data/weiji")

CORE_VARS = ["U", "V", "T", "OMEGA", "PS", "Z3", "O3"]

RUN2_START_YEAR = 105
SHIFT_YEARS = 104
EXPECTED_NTIME = 365

# None = 自动处理所有符合结构的实验
# 示例：["B2000WCN"] 只处理名字里带 B2000WCN 的实验
INCLUDE_PATTERNS = None

DRY_RUN = False
OVERWRITE_TIMEFIX_OUTPUT = False
OVERWRITE_EXISTING_BACKUP = False

# 这些实验的文件名年份已经是目标连续年份，但内部 date/time 仍保留各 run 的原始年份。
# 对它们不使用固定 SHIFT_YEARS，而是按文件名年份重建 date/time/time_bnds。
FILENAME_YEAR_TIMEFIX_EXPERIMENTS = {"B2000WCN007009010011_Clim3D"}

MONTH_LENGTHS_NOLEAP = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]

print("BASE_ROOT =", BASE_ROOT)
print("RUN2_START_YEAR =", RUN2_START_YEAR)
print("SHIFT_YEARS =", SHIFT_YEARS)
print("EXPECTED_NTIME =", EXPECTED_NTIME)
print("INCLUDE_PATTERNS =", INCLUDE_PATTERNS)
print("DRY_RUN =", DRY_RUN)
print("OVERWRITE_TIMEFIX_OUTPUT =", OVERWRITE_TIMEFIX_OUTPUT)
print("OVERWRITE_EXISTING_BACKUP =", OVERWRITE_EXISTING_BACKUP)
print("FILENAME_YEAR_TIMEFIX_EXPERIMENTS =", sorted(FILENAME_YEAR_TIMEFIX_EXPERIMENTS))


# ============================================================
# Helpers
# ============================================================

def open_nc(path):
    try:
        return xr.open_dataset(path, decode_times=False, engine="netcdf4")
    except Exception:
        return xr.open_dataset(path, decode_times=False)

def parse_file_year(path, var):
    pat = rf".*\.cam\.h3\.([0-9]{{4}})\.{re.escape(var)}\.nc$"
    m = re.match(pat, path.name)
    return int(m.group(1)) if m else None

def date_year(date_value):
    try:
        return int(date_value) // 10000
    except Exception:
        return None

def date_to_mmdd(date_int):
    return int(date_int) % 10000

def date_to_doy(date_int):
    mmdd = date_to_mmdd(date_int)
    m = mmdd // 100
    d = mmdd % 100
    return sum(MONTH_LENGTHS_NOLEAP[:m - 1]) + d

def make_full_dates(year):
    dates = []
    doys = []
    for m, ndays in enumerate(MONTH_LENGTHS_NOLEAP, start=1):
        for d in range(1, ndays + 1):
            dates.append(year * 10000 + m * 100 + d)
            doys.append(sum(MONTH_LENGTHS_NOLEAP[:m - 1]) + d)
    return np.asarray(dates, dtype=np.int32), np.asarray(doys, dtype=np.int32)

def make_full_time(year):
    _, doys = make_full_dates(year)
    return ((year - 1) * 365 + (doys - 1)).astype(np.float64)

def make_full_time_bnds(full_time):
    tb = np.empty((full_time.size, 2), dtype=np.float64)
    tb[:, 0] = full_time - 1.0
    tb[:, 1] = full_time
    return tb

def make_time_from_dates_for_year(dates, target_year):
    doys = np.asarray([date_to_doy(d) for d in dates], dtype=np.int32)
    return ((target_year - 1) * 365 + (doys - 1)).astype(np.float64)

def build_reindex_key_from_existing_dates(old_dates, target_year):
    mapped = []
    for d in old_dates:
        mmdd = date_to_mmdd(d)
        mapped.append(target_year * 10000 + mmdd)
    return np.asarray(mapped, dtype=np.int32)

def safe_float(x):
    if x is None:
        return ""
    try:
        return float(x)
    except Exception:
        return str(x)

def safe_int(x):
    if x is None:
        return ""
    try:
        return int(x)
    except Exception:
        return str(x)

def first_last(arr):
    arr = np.asarray(arr)
    if arr.size == 0:
        return None, None
    flat = arr.reshape(-1)
    return flat[0], flat[-1]

def make_encoding(ds, fill_float_nan=False):
    encoding = {}
    for name in ds.variables:
        enc = {}
        if name in ds.data_vars:
            enc["zlib"] = True
            enc["complevel"] = 1

        if fill_float_nan and np.issubdtype(ds[name].dtype, np.floating):
            enc["_FillValue"] = np.nan
        else:
            old_fv = ds[name].encoding.get("_FillValue", None)
            if old_fv is not None:
                enc["_FillValue"] = old_fv

        if enc:
            encoding[name] = enc
    return encoding


# ============================================================
# Auto-detect experiments
# ============================================================

def is_experiment_root(path):
    if not path.is_dir():
        return False
    if path.name.endswith("_timefixed"):
        return False
    if path.name.startswith("."):
        return False
    if path.name == "_time_diagnostics":
        return False

    if INCLUDE_PATTERNS is not None:
        if not any(pat in path.name for pat in INCLUDE_PATTERNS):
            return False

    # 必须有这些变量目录
    for var in CORE_VARS:
        if not (path / var).is_dir():
            return False

    # 至少有一个符合模式的 nc 文件
    for var in CORE_VARS:
        files = sorted((path / var).glob(f"*.{var}.nc"))
        for f in files:
            if ".cam.h3." in f.name:
                return True

    return False

def detect_experiment_roots(base_root):
    roots = []
    for p in sorted(base_root.iterdir()):
        if is_experiment_root(p):
            roots.append(p)
    return roots


# ============================================================
# Time-fix stage
# ============================================================

def rebuild_existing_time_to_filename_year(ds, year):
    if "time" not in ds:
        raise ValueError("missing time variable")
    if "date" not in ds:
        raise ValueError("missing date variable")

    old_dates = ds["date"].values.astype(np.int64)
    new_dates = build_reindex_key_from_existing_dates(old_dates, year)
    new_time = make_time_from_dates_for_year(old_dates, year)

    ds = ds.assign_coords(time=("time", new_time))
    ds["time"].attrs = ds["time"].attrs.copy()
    ds["time"].attrs["units"] = "days since 0001-01-01 00:00:00"
    ds["time"].attrs["calendar"] = "noleap"
    if "time_bnds" in ds:
        ds["time"].attrs["bounds"] = "time_bnds"

    ds["date"] = xr.DataArray(new_dates, dims=("time",), attrs=ds["date"].attrs.copy())

    if "time_bnds" in ds:
        tb_dims = ds["time_bnds"].dims
        tb_attrs = ds["time_bnds"].attrs.copy()
        ds["time_bnds"] = xr.DataArray(make_full_time_bnds(new_time), dims=tb_dims, attrs=tb_attrs)

    return ds


def timefix_one_file(in_file, out_file, var, mode="shift_run2"):
    year = parse_file_year(in_file, var)
    if year is None:
        raise ValueError(f"Cannot parse year from {in_file.name}")

    if out_file.exists() and out_file.stat().st_size > 0 and not OVERWRITE_TIMEFIX_OUTPUT:
        return {"file": in_file.name, "var": var, "year": year, "action": "skip_existing"}

    if DRY_RUN:
        if mode == "filename_year":
            action = "rebuild_filename_year"
        else:
            action = "copy" if year < RUN2_START_YEAR else "shift"
        return {"file": in_file.name, "var": var, "year": year, "action": f"dry_run_{action}"}

    out_file.parent.mkdir(parents=True, exist_ok=True)

    if mode == "filename_year":
        ds = open_nc(in_file)
        try:
            ds = rebuild_existing_time_to_filename_year(ds, year)
        except Exception:
            ds.close()
            raise

        tmp = out_file.with_suffix(out_file.suffix + ".tmp")
        ds.to_netcdf(tmp, format="NETCDF4", engine="netcdf4", encoding=make_encoding(ds, fill_float_nan=False))
        ds.close()
        tmp.replace(out_file)
        return {"file": in_file.name, "var": var, "year": year, "action": "rebuilt_filename_year"}

    if year < RUN2_START_YEAR:
        shutil.copy2(in_file, out_file)
        return {"file": in_file.name, "var": var, "year": year, "action": "copied"}

    shift_days = SHIFT_YEARS * 365
    shift_date = SHIFT_YEARS * 10000

    ds = open_nc(in_file)

    # shift time
    if "time" not in ds:
        ds.close()
        raise ValueError(f"{in_file.name}: missing time variable")

    old_time = ds["time"].values.astype(np.float64)
    new_time = old_time + shift_days
    ds = ds.assign_coords(time=("time", new_time))
    ds["time"].attrs = ds["time"].attrs.copy()
    ds["time"].attrs["units"] = "days since 0001-01-01 00:00:00"
    ds["time"].attrs["calendar"] = "noleap"
    if "time_bnds" in ds:
        ds["time"].attrs["bounds"] = "time_bnds"

    # shift date
    if "date" in ds:
        old_date = ds["date"].values.astype(np.int64)
        ds["date"] = xr.DataArray(old_date + shift_date, dims=("time",), attrs=ds["date"].attrs.copy())

    # rebuild time_bnds
    if "time_bnds" in ds:
        tb_dims = ds["time_bnds"].dims
        tb_attrs = ds["time_bnds"].attrs.copy()
        new_tb = np.empty((new_time.size, 2), dtype=np.float64)
        new_tb[:, 0] = new_time - 1.0
        new_tb[:, 1] = new_time
        ds["time_bnds"] = xr.DataArray(new_tb, dims=tb_dims, attrs=tb_attrs)

    tmp = out_file.with_suffix(out_file.suffix + ".tmp")
    ds.to_netcdf(tmp, format="NETCDF4", engine="netcdf4", encoding=make_encoding(ds, fill_float_nan=False))
    ds.close()
    tmp.replace(out_file)

    return {"file": in_file.name, "var": var, "year": year, "action": "shifted"}


def run_timefix(exp_root, out_root):
    out_root.mkdir(parents=True, exist_ok=True)
    for var in CORE_VARS:
        (out_root / var).mkdir(parents=True, exist_ok=True)

    marker = out_root / ".TIMEFIX_DONE"
    records = []

    if marker.exists():
        print(f"[SKIP TIMEFIX] marker exists: {marker}")
        return pd.DataFrame(records)

    mode = "filename_year" if exp_root.name in FILENAME_YEAR_TIMEFIX_EXPERIMENTS else "shift_run2"

    print()
    print("=" * 100)
    print(f"TIMEFIX: {exp_root.name}")
    print("=" * 100)
    print(f"[TIMEFIX MODE] {mode}")

    for var in CORE_VARS:
        files = sorted((exp_root / var).glob(f"*.{var}.nc"))
        print(f"[TIMEFIX SCAN] {var}: {len(files)} files")
        for in_file in files:
            out_file = out_root / var / in_file.name
            rec = timefix_one_file(in_file, out_file, var, mode=mode)
            records.append(rec)

    if not DRY_RUN:
        marker.write_text("TIMEFIX_DONE\n", encoding="utf-8")

    return pd.DataFrame(records)


# ============================================================
# Fill stage
# ============================================================

def fill_one_file(path, var):
    year = parse_file_year(path, var)
    ds = open_nc(path)

    if "time" not in ds.dims:
        ds.close()
        return {"file": path.name, "var": var, "year": year, "action": "skip_no_time", "missing_days": None}, None

    ntime = ds.sizes["time"]
    if ntime == EXPECTED_NTIME:
        ds.close()
        return {"file": path.name, "var": var, "year": year, "action": "skip_full", "missing_days": 0}, None

    if "date" not in ds:
        ds.close()
        raise ValueError(f"{path.name}: no date variable; cannot fill")

    old_dates_raw = ds["date"].values.astype(np.int64)
    old_dates_mapped = build_reindex_key_from_existing_dates(old_dates_raw, year)

    if len(np.unique(old_dates_mapped)) != len(old_dates_mapped):
        ds.close()
        raise ValueError(f"{path.name}: duplicate mapped dates detected")

    full_dates, _ = make_full_dates(year)
    full_time = make_full_time(year)
    full_time_bnds = make_full_time_bnds(full_time)

    old_set = set(old_dates_mapped.tolist())
    full_set = set(full_dates.tolist())
    missing_dates = np.array(sorted(full_set - old_set), dtype=np.int32)

    print(f"[FILL] {path.name}: {ntime} -> {EXPECTED_NTIME}, missing={len(missing_dates)}")

    if DRY_RUN:
        ds.close()
        return {
            "file": path.name, "var": var, "year": year,
            "action": "dry_run_fill", "missing_days": len(missing_dates)
        }, None

    # preserve attrs
    time_attrs = ds["time"].attrs.copy() if "time" in ds else {}
    date_attrs = ds["date"].attrs.copy() if "date" in ds else {}
    datesec_attrs = ds["datesec"].attrs.copy() if "datesec" in ds else {}
    time_bnds_attrs = ds["time_bnds"].attrs.copy() if "time_bnds" in ds else {}

    ds_idx = ds.assign_coords(time=("time", old_dates_mapped))
    ds_fill = ds_idx.reindex(time=full_dates)

    # rebuild time/date/datesec/time_bnds
    ds_fill = ds_fill.assign_coords(time=("time", full_time))
    ds_fill["time"].attrs = time_attrs
    ds_fill["time"].attrs["units"] = "days since 0001-01-01 00:00:00"
    ds_fill["time"].attrs["calendar"] = "noleap"
    ds_fill["time"].attrs["bounds"] = "time_bnds"

    ds_fill["date"] = xr.DataArray(full_dates.astype(np.int32), dims=("time",), attrs=date_attrs)

    if "datesec" in ds:
        old_datesec = ds["datesec"].values
        datesec_fill_value = int(old_datesec[0]) if old_datesec.size else 0
    else:
        datesec_fill_value = 0

    ds_fill["datesec"] = xr.DataArray(
        np.full(EXPECTED_NTIME, datesec_fill_value, dtype=np.int32),
        dims=("time",),
        attrs=datesec_attrs,
    )

    if "time_bnds" in ds:
        tb_dims = ds["time_bnds"].dims
        if len(tb_dims) != 2:
            ds.close()
            raise ValueError(f"{path.name}: unexpected time_bnds dims {tb_dims}")
        bnd_dim = tb_dims[1]
    else:
        bnd_dim = "nbnd"

    ds_fill["time_bnds"] = xr.DataArray(
        full_time_bnds,
        dims=("time", bnd_dim),
        attrs=time_bnds_attrs,
    )

    backup = path.with_name(path.name + ".bak_before_fill365")
    if backup.exists():
        if OVERWRITE_EXISTING_BACKUP:
            backup.unlink()
            shutil.copy2(path, backup)
    else:
        shutil.copy2(path, backup)

    tmp = path.with_name(path.name + ".tmp_fill365")
    ds_fill.to_netcdf(tmp, format="NETCDF4", engine="netcdf4", encoding=make_encoding(ds_fill, fill_float_nan=True))
    ds.close()
    ds_fill.close()
    tmp.replace(path)

    # NaN verification only on newly inserted dates
    ds_out = open_nc(path)
    nan_rec = None
    if len(missing_dates) > 0:
        out_dates = ds_out["date"].values.astype(np.int64)
        idx = np.where(np.isin(out_dates, missing_dates))[0]
        arr = ds_out[var].isel(time=idx).values
        if np.issubdtype(arr.dtype, np.floating):
            all_nan = bool(np.isnan(arr).all())
            non_nan_count = int(np.size(arr) - np.isnan(arr).sum())
        else:
            all_nan = False
            non_nan_count = int(np.size(arr))
        nan_rec = {
            "file": path.name,
            "var": var,
            "year": year,
            "missing_days": len(missing_dates),
            "checked_points": int(np.size(arr)),
            "all_inserted_values_nan": all_nan,
            "non_nan_inserted_points": non_nan_count,
        }
    ds_out.close()

    return {
        "file": path.name,
        "var": var,
        "year": year,
        "action": "filled",
        "missing_days": len(missing_dates)
    }, nan_rec


def run_fill(out_root):
    marker = out_root / ".FILL365_DONE"
    fill_records = []
    nan_records = []

    if marker.exists():
        print(f"[SKIP FILL] marker exists: {marker}")
        return pd.DataFrame(fill_records), pd.DataFrame(nan_records)

    print()
    print("=" * 100)
    print(f"FILL365: {out_root.name}")
    print("=" * 100)

    for var in CORE_VARS:
        files = sorted((out_root / var).glob(f"*.{var}.nc"))
        print(f"[FILL SCAN] {var}: {len(files)} files")
        for f in files:
            rec, nan_rec = fill_one_file(f, var)
            fill_records.append(rec)
            if nan_rec is not None:
                nan_records.append(nan_rec)

    if not DRY_RUN:
        marker.write_text("FILL365_DONE\n", encoding="utf-8")

    return pd.DataFrame(fill_records), pd.DataFrame(nan_records)


# ============================================================
# Validation
# ============================================================

def inspect_file(path, var):
    fname_year = parse_file_year(path, var)

    row = {
        "var": var,
        "file": str(path),
        "fname": path.name,
        "fname_year": fname_year if fname_year is not None else "",
        "ntime": "",
        "time_first": "",
        "time_last": "",
        "date_first": "",
        "date_last": "",
        "date_first_year": "",
        "date_last_year": "",
        "datesec_first": "",
        "datesec_last": "",
        "time_bnds_first_left": "",
        "time_bnds_first_right": "",
        "time_bnds_last_left": "",
        "time_bnds_last_right": "",
        "time_units": "",
        "time_calendar": "",
        "time_expected_first": "",
        "time_expected_last": "",
        "time_mismatch": "",
        "partial_year": "",
        "year_mismatch": "",
        "error": "",
    }

    try:
        ds = open_nc(path)

        row["ntime"] = ds.sizes.get("time", "")

        if "time" in ds:
            row["time_units"] = ds["time"].attrs.get("units", "")
            row["time_calendar"] = ds["time"].attrs.get("calendar", "")
            t0, t1 = first_last(ds["time"].values)
            row["time_first"] = safe_float(t0)
            row["time_last"] = safe_float(t1)
            if fname_year is not None and t0 is not None and t1 is not None:
                expected_t0 = float((fname_year - 1) * 365)
                expected_t1 = float(fname_year * 365 - 1)
                row["time_expected_first"] = expected_t0
                row["time_expected_last"] = expected_t1
                row["time_mismatch"] = "YES" if (float(t0) != expected_t0 or float(t1) != expected_t1) else "no"

        if "date" in ds:
            d0, d1 = first_last(ds["date"].values)
            row["date_first"] = safe_int(d0)
            row["date_last"] = safe_int(d1)
            y0 = date_year(d0)
            y1 = date_year(d1)
            row["date_first_year"] = y0 if y0 is not None else ""
            row["date_last_year"] = y1 if y1 is not None else ""
            if fname_year is not None and y0 is not None and y1 is not None:
                row["year_mismatch"] = "YES" if (y0 != fname_year or y1 != fname_year) else "no"

        if "datesec" in ds:
            s0, s1 = first_last(ds["datesec"].values)
            row["datesec_first"] = safe_int(s0)
            row["datesec_last"] = safe_int(s1)

        if "time_bnds" in ds:
            tb = np.asarray(ds["time_bnds"].values)
            if tb.ndim == 2 and tb.shape[0] > 0:
                row["time_bnds_first_left"] = safe_float(tb[0, 0])
                row["time_bnds_first_right"] = safe_float(tb[0, -1])
                row["time_bnds_last_left"] = safe_float(tb[-1, 0])
                row["time_bnds_last_right"] = safe_float(tb[-1, -1])

        if row["ntime"] != "" and int(row["ntime"]) != EXPECTED_NTIME:
            row["partial_year"] = "YES"
        else:
            row["partial_year"] = "no"

        ds.close()

    except Exception as e:
        row["error"] = repr(e)

    return row


def run_validation(out_root, fill_df=None, nan_df=None):
    diag_dir = out_root / "_time_diagnostics"
    diag_dir.mkdir(parents=True, exist_ok=True)

    rows = []
    for var in CORE_VARS:
        files = sorted((out_root / var).glob(f"*.{var}.nc"))
        for f in files:
            rows.append(inspect_file(f, var))

    df = pd.DataFrame(rows)
    summary_csv = diag_dir / "time_debug_summary.csv"
    df.to_csv(summary_csv, index=False)

    coverage_rows = []
    for var in CORE_VARS:
        sub = df[(df["var"] == var) & (df["fname_year"] != "")].copy()
        if len(sub) == 0:
            continue
        sub["fname_year_int"] = sub["fname_year"].astype(int)
        years = sub["fname_year_int"].sort_values().values
        coverage_rows.append({
            "var": var,
            "nfiles": len(sub),
            "first_year": sub["fname_year_int"].min(),
            "last_year": sub["fname_year_int"].max(),
            "n_year_gaps": int((np.diff(years) != 1).sum()) if len(years) > 1 else 0,
            "n_partial": int((sub["partial_year"] == "YES").sum()),
            "n_mismatch": int((sub["year_mismatch"] == "YES").sum()),
            "n_time_mismatch": int((sub["time_mismatch"] == "YES").sum()),
            "n_error": int((sub["error"] != "").sum()),
        })
    coverage_df = pd.DataFrame(coverage_rows)

    partial_df = df[df["partial_year"] == "YES"].copy()
    mismatch_df = df[df["year_mismatch"] == "YES"].copy()

    transition_rows = []
    for var in CORE_VARS:
        sub = df[(df["var"] == var) & (df["fname_year"] != "")].copy()
        if len(sub) == 0:
            continue
        sub["fname_year_int"] = sub["fname_year"].astype(int)
        sub = sub.sort_values("fname_year_int")

        run1 = sub[sub["fname_year_int"] < RUN2_START_YEAR]
        run2 = sub[sub["fname_year_int"] >= RUN2_START_YEAR]
        if len(run1) == 0 or len(run2) == 0:
            continue

        a = run1.iloc[-1]
        b = run2.iloc[0]

        try:
            dt = float(b["time_first"]) - float(a["time_last"])
        except Exception:
            dt = np.nan

        transition_rows.append({
            "var": var,
            "last_run1_file_year": a["fname_year"],
            "last_run1_ntime": a["ntime"],
            "last_run1_date_last": a["date_last"],
            "last_run1_time_last": a["time_last"],
            "first_run2_file_year": b["fname_year"],
            "first_run2_ntime": b["ntime"],
            "first_run2_date_first": b["date_first"],
            "first_run2_time_first": b["time_first"],
            "delta_time_first2_minus_last1": dt,
        })

    transition_df = pd.DataFrame(transition_rows)
    transition_csv = diag_dir / "time_debug_transition.csv"
    transition_df.to_csv(transition_csv, index=False)

    all_transition_rows = []
    for var in CORE_VARS:
        sub = df[(df["var"] == var) & (df["fname_year"] != "")].copy()
        if len(sub) <= 1:
            continue
        sub["fname_year_int"] = sub["fname_year"].astype(int)
        sub = sub.sort_values("fname_year_int")
        for i in range(len(sub) - 1):
            a = sub.iloc[i]
            b = sub.iloc[i + 1]
            try:
                dt = float(b["time_first"]) - float(a["time_last"])
            except Exception:
                dt = np.nan
            all_transition_rows.append({
                "var": var,
                "from_year": a["fname_year"],
                "to_year": b["fname_year"],
                "delta_time_next_first_minus_this_last": dt,
                "ok": bool(np.isclose(dt, 1.0)),
            })

    all_transition_df = pd.DataFrame(all_transition_rows)
    all_transition_csv = diag_dir / "time_debug_transition_all_years.csv"
    all_transition_df.to_csv(all_transition_csv, index=False)

    if fill_df is not None:
        fill_df.to_csv(diag_dir / "fill365_summary.csv", index=False)
    if nan_df is not None:
        nan_df.to_csv(diag_dir / "fill365_nan_check.csv", index=False)

    print()
    print("=" * 100)
    print(f"VALIDATION: {out_root.name}")
    print("=" * 100)
    print(coverage_df.to_string(index=False))

    if len(partial_df):
        print()
        print("PARTIAL FILES:")
        print(partial_df[["var", "fname_year", "ntime", "date_first", "date_last", "fname"]].to_string(index=False))

    if len(mismatch_df):
        print()
        print("MISMATCH FILES:")
        print(mismatch_df[["var", "fname_year", "date_first", "date_last", "fname"]].to_string(index=False))

    if len(transition_df):
        print()
        print("TRANSITION:")
        print(transition_df.to_string(index=False))

    if len(all_transition_df) and not bool(all_transition_df["ok"].all()):
        print()
        print("BAD ALL-YEAR TRANSITIONS:")
        print(all_transition_df[~all_transition_df["ok"]].to_string(index=False))

    ok_gaps = int((coverage_df["n_year_gaps"] == 0).all()) if len(coverage_df) else 1
    ok_partial = int((coverage_df["n_partial"] == 0).all()) if len(coverage_df) else 1
    ok_mismatch = int((coverage_df["n_mismatch"] == 0).all()) if len(coverage_df) else 1
    ok_time = int((coverage_df["n_time_mismatch"] == 0).all()) if len(coverage_df) else 1
    ok_error = int((coverage_df["n_error"] == 0).all()) if len(coverage_df) else 1

    ok_transition = 1
    if len(transition_df):
        try:
            ok_transition = int(np.allclose(transition_df["delta_time_first2_minus_last1"].astype(float).values, 1.0))
        except Exception:
            ok_transition = 0

    ok_all_transitions = 1
    if len(all_transition_df):
        ok_all_transitions = int(bool(all_transition_df["ok"].all()))

    ok_nan = 1
    if nan_df is not None and len(nan_df):
        ok_nan = int(bool(nan_df["all_inserted_values_nan"].all()))

    all_ok = bool(ok_gaps and ok_partial and ok_mismatch and ok_time and ok_error and ok_transition and ok_all_transitions and ok_nan)

    report = {
        "experiment": out_root.name,
        "summary_csv": str(summary_csv),
        "transition_csv": str(transition_csv),
        "all_transition_csv": str(all_transition_csv),
        "ok_gaps": bool(ok_gaps),
        "ok_partial": bool(ok_partial),
        "ok_mismatch": bool(ok_mismatch),
        "ok_time": bool(ok_time),
        "ok_error": bool(ok_error),
        "ok_transition": bool(ok_transition),
        "ok_all_transitions": bool(ok_all_transitions),
        "ok_nan": bool(ok_nan),
        "all_ok": bool(all_ok),
    }

    with open(diag_dir / "validation_report.json", "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)

    return report


# ============================================================
# Main
# ============================================================

exp_roots = detect_experiment_roots(BASE_ROOT)

print()
print("#" * 100)
print("Detected experiments")
print("#" * 100)
for p in exp_roots:
    print(p)

if not exp_roots:
    raise SystemExit("No experiment roots detected.")

for exp_root in exp_roots:
    out_root = exp_root.with_name(exp_root.name + "_timefixed")
    final_marker = out_root / ".TIMEFIX_AND_FILL_DONE"

    print()
    print("#" * 120)
    print(f"PROCESSING: {exp_root.name}")
    print(f"IN  = {exp_root}")
    print(f"OUT = {out_root}")
    print("#" * 120)

    if final_marker.exists():
        print(f"[SKIP EXPERIMENT] final marker exists: {final_marker}")
        continue

    timefix_df = run_timefix(exp_root, out_root)
    fill_df, nan_df = run_fill(out_root)
    report = run_validation(out_root, fill_df=fill_df, nan_df=nan_df)

    if report["all_ok"] and not DRY_RUN:
        final_marker.write_text("TIMEFIX_AND_FILL_DONE\n", encoding="utf-8")
        print(f"[DONE] marker written: {final_marker}")
    else:
        print("[WARN] final marker not written because validation did not fully pass.")
        print(report)

print()
print("ALL EXPERIMENTS FINISHED.")